In [0]:
# Use your existing catalog
CATALOG = "healthcare-org-catalog"

# Dedicated schema for pytest
TEST_SCHEMA = "pytest_schema"

FULL_SCHEMA = f"{CATALOG}.{TEST_SCHEMA}"

print(FULL_SCHEMA)

healthcare-org-catalog.pytest_schema


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{TEST_SCHEMA}`
COMMENT 'Schema for running pytest validations'
""")

print(f"✅ Schema {FULL_SCHEMA} created successfully")

✅ Schema healthcare-org-catalog.pytest_schema created successfully


In [0]:
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{TEST_SCHEMA}`")

print("✅ Using pytest schema")

✅ Using pytest schema


In [0]:
import pytest
from pyspark.sql import SparkSession

CATALOG = "`healthcare-org-catalog`"
TEST_SCHEMA = "`pytest_schema`"

FULL_SCHEMA = f"{CATALOG}.{TEST_SCHEMA}"

BRONZE_TABLES = [
    "hospital",
    "demographics",
    "diagnosis",
    "lab",
    "vitals"
]

# --------------------------------------------------
# Spark Session
# --------------------------------------------------

def get_spark():
    return SparkSession.builder.appName("bronze-tests").getOrCreate()


# --------------------------------------------------
# Test 1 : Tables Exist
# --------------------------------------------------

def test_bronze_tables_exist(spark):

    print("\n🔍 Checking Bronze Tables in PYTEST SCHEMA...\n")

    tables = spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}")
    table_list = [row.tableName.lower() for row in tables.collect()]

    for table in BRONZE_TABLES:
        print(f"➡️ Checking {table}")
        assert table.lower() in table_list, f"❌ {table} missing"

    print("\n✅ All Bronze tables exist\n")


# --------------------------------------------------
# Test 2 : Tables Have Data
# --------------------------------------------------

def test_bronze_tables_have_data(spark):

    for table in BRONZE_TABLES:

        print(f"\n📊 Checking data: {table}")

        df = spark.table(f"{FULL_SCHEMA}.{table}")
        count = df.count()

        print(f"Rows: {count}")

        assert count > 0, f"❌ {table} empty"

        print(f"✅ {table} passed\n")


# --------------------------------------------------
# 🔥 MAIN RUNNER (IMPORTANT)
# --------------------------------------------------

if __name__ == "__main__":

    spark = get_spark()

    print("\n🚀 Running Bronze Tests...\n")

    test_bronze_tables_exist(spark)
    test_bronze_tables_have_data(spark)

    print("\n🎯 All Bronze Tests Passed Successfully!\n")


🚀 Running Bronze Tests...


🔍 Checking Bronze Tables in PYTEST SCHEMA...

➡️ Checking hospital
➡️ Checking demographics
➡️ Checking diagnosis
➡️ Checking lab
➡️ Checking vitals

✅ All Bronze tables exist


📊 Checking data: hospital
Rows: 50300
✅ hospital passed


📊 Checking data: demographics
Rows: 50099
✅ demographics passed


📊 Checking data: diagnosis
Rows: 50100
✅ diagnosis passed


📊 Checking data: lab
Rows: 50099
✅ lab passed


📊 Checking data: vitals
Rows: 50097
✅ vitals passed


🎯 All Bronze Tests Passed Successfully!



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# --------------------------------------------------
# Spark Session
# --------------------------------------------------

def get_spark():
    return SparkSession.builder.appName("silver-layer-build").getOrCreate()


# --------------------------------------------------
# Catalog + Schemas
# --------------------------------------------------

CATALOG = "`healthcare-org-catalog`"
BRONZE_SCHEMA = "`healthcare-org-schema`"
SILVER_SCHEMA = "`silver_layer_schema`"

FULL_BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}"
FULL_SILVER = f"{CATALOG}.{SILVER_SCHEMA}"


# --------------------------------------------------
# Helper : Drop table if exists
# --------------------------------------------------

def drop_table_if_exists(spark, table_name):
    print(f"🗑️ Dropping table if exists: {table_name}")
    spark.sql(f"DROP TABLE IF EXISTS {FULL_SILVER}.{table_name}")


# --------------------------------------------------
# Helper : Write Delta Table
# --------------------------------------------------

def write_silver(spark, df, table_name):

    print(f"\n⚙️ Processing: {table_name}")

    before_count = df.count()
    print(f"📥 Input rows: {before_count}")

    drop_table_if_exists(spark, table_name)

    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{FULL_SILVER}.{table_name}")

    after_count = spark.table(f"{FULL_SILVER}.{table_name}").count()

    print(f"📤 Output rows: {after_count}")
    print(f"✅ Written: {table_name}\n")


# --------------------------------------------------
# 1️⃣ silver_hospital
# --------------------------------------------------

def build_silver_hospital(spark):

    print("\n🏥 Building silver_hospital...")

    df = spark.table(f"{FULL_BRONZE}.hospital")

    df = df.filter(col("hospital_id").isNotNull())

    write_silver(spark, df, "silver_hospital")


# --------------------------------------------------
# 2️⃣ silver_demographics
# --------------------------------------------------

def build_silver_demographics(spark):

    print("\n👤 Building silver_demographics...")

    df = spark.table(f"{FULL_BRONZE}.demographics")

    df = df.filter(col("patient_id").isNotNull())

    df = df.filter(
        (col("age") >= 0) &
        (col("age") <= 120)
    )

    print("🔄 Applying deduplication...")

    if "updated_at" in [c.lower() for c in df.columns]:

        window = Window.partitionBy("patient_id").orderBy(col("updated_at").desc())

        df = (
            df.withColumn("rn", row_number().over(window))
            .filter(col("rn") == 1)
            .drop("rn")
        )
        print("✅ Dedup using updated_at")

    else:
        df = df.dropDuplicates(["patient_id"])
        print("✅ Dedup using dropDuplicates")

    write_silver(spark, df, "silver_demographics")


# --------------------------------------------------
# 3️⃣ silver_diagnosis
# --------------------------------------------------

def build_silver_diagnosis(spark):

    print("\n🧾 Building silver_diagnosis...")

    df = spark.table(f"{FULL_BRONZE}.diagnosis")

    df = df.filter(col("patient_id").isNotNull())

    write_silver(spark, df, "silver_diagnosis")


# --------------------------------------------------
# 4️⃣ silver_lab
# --------------------------------------------------

def build_silver_lab(spark):

    print("\n🧪 Building silver_lab...")

    df = spark.table(f"{FULL_BRONZE}.lab")

    df = df.filter(col("patient_id").isNotNull())

    write_silver(spark, df, "silver_lab")


# --------------------------------------------------
# 5️⃣ silver_vitals
# --------------------------------------------------

def build_silver_vitals(spark):

    print("\n💓 Building silver_vitals...")

    df = spark.table(f"{FULL_BRONZE}.vitals")

    df = df.filter(col("patient_id").isNotNull())

    if "heart_rate" in [c.lower() for c in df.columns]:

        print("❤️ Filtering heart_rate")

        df = df.filter(
            col("heart_rate").isNotNull() &
            (col("heart_rate") >= 30) &
            (col("heart_rate") <= 200)
        )

    if "blood_pressure_sys" in df.columns:
        print("🩸 Filtering blood_pressure_sys")
        df = df.filter(col("blood_pressure_sys").isNotNull())

    write_silver(spark, df, "silver_vitals")


# --------------------------------------------------
# 🔥 MAIN RUNNER (LIKE BRONZE)
# --------------------------------------------------

if __name__ == "__main__":

    spark = get_spark()

    print("\n🚀 Starting Silver Layer Pipeline...\n")

    build_silver_hospital(spark)
    build_silver_demographics(spark)
    build_silver_diagnosis(spark)
    build_silver_lab(spark)
    build_silver_vitals(spark)

    print("\n🎯 All Silver Tables Built Successfully!\n")


🚀 Starting Silver Layer Pipeline...


🏥 Building silver_hospital...

⚙️ Processing: silver_hospital
📥 Input rows: 50300
🗑️ Dropping table if exists: silver_hospital
📤 Output rows: 50300
✅ Written: silver_hospital


👤 Building silver_demographics...
🔄 Applying deduplication...
✅ Dedup using dropDuplicates

⚙️ Processing: silver_demographics
📥 Input rows: 40967
🗑️ Dropping table if exists: silver_demographics
📤 Output rows: 40967
✅ Written: silver_demographics


🧾 Building silver_diagnosis...

⚙️ Processing: silver_diagnosis
📥 Input rows: 50100
🗑️ Dropping table if exists: silver_diagnosis
📤 Output rows: 50100
✅ Written: silver_diagnosis


🧪 Building silver_lab...

⚙️ Processing: silver_lab
📥 Input rows: 50099
🗑️ Dropping table if exists: silver_lab
📤 Output rows: 50099
✅ Written: silver_lab


💓 Building silver_vitals...
❤️ Filtering heart_rate
🩸 Filtering blood_pressure_sys

⚙️ Processing: silver_vitals
📥 Input rows: 50085
🗑️ Dropping table if exists: silver_vitals
📤 Output rows: 50085

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# --------------------------------------------------
# Spark Session
# --------------------------------------------------

def get_spark():
    return SparkSession.builder.appName("gold-tests").getOrCreate()


# --------------------------------------------------
# Catalog + Schema
# --------------------------------------------------

CATALOG = "`healthcare-org-catalog`"
GOLD_SCHEMA = "`gold_layer_schema`"

FULL_GOLD = f"{CATALOG}.{GOLD_SCHEMA}"

DIM_PATIENT = f"{FULL_GOLD}.dim_patient"
DIM_HOSPITAL = f"{FULL_GOLD}.dim_hospital"
DIM_DOCTOR = f"{FULL_GOLD}.dim_doctor"
DIM_DATE = f"{FULL_GOLD}.dim_date"

FACT_TABLE = f"{FULL_GOLD}.fact_patient_health_metrics"


# --------------------------------------------------
# Test 1 : dim_patient exists
# --------------------------------------------------

def test_dim_patient_exists(spark):

    print("\n🔍 Checking dim_patient existence...")

    tables = spark.sql(f"SHOW TABLES IN {FULL_GOLD}")

    exists = tables.filter(tables.tableName == "dim_patient").count()

    print(f"Found: {exists}")

    assert exists == 1, "❌ dim_patient missing"

    print("✅ dim_patient exists\n")


# --------------------------------------------------
# Test 2 : dim_patient not empty
# --------------------------------------------------

def test_dim_patient_not_empty(spark):

    print("\n📊 Checking dim_patient data...")

    df = spark.table(DIM_PATIENT)
    count = df.count()

    print(f"Rows: {count}")

    assert count > 0, "❌ dim_patient is empty"

    print("✅ dim_patient has data\n")


# --------------------------------------------------
# Test 3 : patient_id not null
# --------------------------------------------------

def test_patient_id_not_null(spark):

    print("\n🧬 Checking patient_id nulls...")

    df = spark.table(DIM_PATIENT)
    null_ids = df.filter(col("patient_id").isNull()).count()

    print(f"Null count: {null_ids}")

    assert null_ids == 0, "❌ NULL patient_id found"

    print("✅ No NULL patient_id\n")


# --------------------------------------------------
# Test 4 : fact table exists + not empty
# --------------------------------------------------

def test_fact_table_not_empty(spark):

    print("\n📊 Checking fact table...")

    df = spark.table(FACT_TABLE)
    count = df.count()

    print(f"Rows: {count}")

    assert count > 0, "❌ Fact table empty"

    print("✅ Fact table has data\n")


# --------------------------------------------------
# Test 5 : risk_score not null
# --------------------------------------------------

def test_risk_score_not_null(spark):

    print("\n⚠️ Checking risk_score...")

    df = spark.table(FACT_TABLE)
    null_risk = df.filter(col("risk_score").isNull()).count()

    print(f"Null risk_score: {null_risk}")

    assert null_risk == 0, "❌ NULL risk_score found"

    print("✅ risk_score valid\n")


# --------------------------------------------------
# 🔥 MAIN RUNNER (IMPORTANT)
# --------------------------------------------------

if __name__ == "__main__":

    spark = get_spark()

    print("\n🚀 Running Gold Layer Tests...\n")

    test_dim_patient_exists(spark)
    test_dim_patient_not_empty(spark)
    test_patient_id_not_null(spark)
    test_fact_table_not_empty(spark)
    test_risk_score_not_null(spark)

    print("\n🎯 All Gold Tests Passed Successfully!\n")


🚀 Running Gold Layer Tests...


🔍 Checking dim_patient existence...
Found: 1
✅ dim_patient exists


📊 Checking dim_patient data...
Rows: 48403
✅ dim_patient has data


🧬 Checking patient_id nulls...
Null count: 0
✅ No NULL patient_id


📊 Checking fact table...
Rows: 50099
✅ Fact table has data


⚠️ Checking risk_score...
Null risk_score: 0
✅ risk_score valid


🎯 All Gold Tests Passed Successfully!

